# Scholarly stats check

Llama a las funciones reales de `libs.metrics.io` (las que usa el pipeline de
similarity) y compara h-index, i10-index y e-index contra la fórmula escrita a
mano, para ver si coinciden.

In [8]:
import sys
sys.path.insert(0, '../..')

import math
import numpy as np
import pandas as pd

from libs.metrics.similarity import _scholarly_stats_from_citations

pd.set_option('display.width', 200)

## 1. Helper: correr la implementación sobre listas de citas

`_scholarly_stats_from_citations` recibe arrays planos (fila_autor, citas_del_paper),
no listas por autor, así que las aplanamos aquí.

In [9]:
def stats_for(per_author: dict[str, list[int]]) -> pd.DataFrame:
    """{nombre: [citas por paper]} -> DataFrame con h_index / i10_index / e_index."""
    names = list(per_author)
    idx, cits = [], []
    for i, name in enumerate(names):
        idx.extend([i] * len(per_author[name]))
        cits.extend(per_author[name])
    out = _scholarly_stats_from_citations(
        np.array(idx, dtype=np.int64),
        np.array(cits, dtype=np.int64),
        n_authors=len(names),
    )
    out.index = names
    return out

## 2. Fórmulas de referencia (a mano)

- **h-index**: mayor r tal que el paper en posición r (orden descendente) tiene >= r citas.
- **i10-index**: papers con >= 10 citas.
- **e-index**: la definición de LLMScholarBench (`GTBuilder/APS/code/libs/scholar.py`),
  entropía del reparto de citas: `e = -1/N * sum_i c_i * log(c_i / c_total)`,
  donde **N cuenta solo los papers citados** (los no citados no aparecen en la tabla
  de citas y por tanto no entran al denominador).

In [10]:
def ref_h_index(citations):
    c = sorted(citations, reverse=True)
    return sum(1 for r, ci in enumerate(c, start=1) if ci >= r)


def ref_i10_index(citations):
    return sum(1 for c in citations if c >= 10)


def ref_e_index(citations):
    c_total = sum(citations)
    n_cited = sum(1 for c in citations if c > 0)
    if n_cited == 0 or c_total == 0:
        return 0.0
    return -1 / n_cited * sum(c * math.log(c / c_total) for c in citations if c > 0)

## 3. Casos de prueba

In [11]:
CASES = {
    'clasico':           [10, 8, 5, 4, 3],
    'un_paper_famoso':   [100],
    'plano':             [5, 5, 5, 5, 5],
    'sin_citas':         [0, 0, 0],
    'con_no_citados':    [10, 8, 5, 0, 0, 0],
    'concentrado':       [100, 1, 1, 1],
    'repartido':         [26, 26, 26, 25],
    'cola_larga':        [50, 20, 12, 11, 10, 9, 3, 2, 1, 1, 0],
    'desordenado':       [3, 10, 4, 8, 5],   # mismo que 'clasico', otro orden
    'calculado_previo':  [2,4,3]
}

impl = stats_for(CASES)
impl

,h_index,i10_index,e_index
clasico,4,1,9.097267
un_paper_famoso,1,1,-0.000000
plano,5,0,8.047190
sin_citas,0,0,0.000000
con_no_citados,3,1,8.135931
concentrado,1,1,4.215017
repartido,4,4,35.693415
cola_larga,6,5,20.952117
desordenado,4,1,9.097267
calculado_previo,2,0,3.182571


## 4. Comparación implementación vs. referencia

In [12]:
ref = pd.DataFrame(
    {
        'h_index': {k: ref_h_index(v) for k, v in CASES.items()},
        'i10_index': {k: ref_i10_index(v) for k, v in CASES.items()},
        'e_index': {k: ref_e_index(v) for k, v in CASES.items()},
    }
)

cmp = impl.join(ref, lsuffix='_impl', rsuffix='_ref')
for col in ['h_index', 'i10_index', 'e_index']:
    cmp[f'{col}_ok'] = np.isclose(cmp[f'{col}_impl'], cmp[f'{col}_ref'], atol=1e-12)

cmp[sorted(cmp.columns)]

,e_index_impl,e_index_ok,e_index_ref,h_index_impl,h_index_ok,h_index_ref,i10_index_impl,i10_index_ok,i10_index_ref
clasico,9.097267,True,9.097267,4,True,4,1,True,1
un_paper_famoso,-0.000000,True,-0.000000,1,True,1,1,True,1
plano,8.047190,True,8.047190,5,True,5,0,True,0
sin_citas,0.000000,True,0.000000,0,True,0,0,True,0
con_no_citados,8.135931,True,8.135931,3,True,3,1,True,1
concentrado,4.215017,True,4.215017,1,True,1,1,True,1
repartido,35.693415,True,35.693415,4,True,4,4,True,4
cola_larga,20.952117,True,20.952117,6,True,6,5,True,5
desordenado,9.097267,True,9.097267,4,True,4,1,True,1
calculado_previo,3.182571,True,3.182571,2,True,2,0,True,0


In [13]:
ok_cols = [c for c in cmp.columns if c.endswith('_ok')]
if cmp[ok_cols].all().all():
    print('Todo coincide.')
else:
    print('NO coinciden estas filas:')
    display(cmp[~cmp[ok_cols].all(axis=1)])

Todo coincide.


## 5. Chequeo aleatorio (fuzz)

Mismo contraste sobre autores generados al azar, por si algún caso raro se escapa.

In [14]:
rng = np.random.default_rng(0)
random_cases = {
    f'a{i}': rng.integers(0, 60, size=int(rng.integers(1, 40))).tolist()
    for i in range(500)
}

impl_r = stats_for(random_cases)
ref_r = pd.DataFrame(
    {
        'h_index': {k: ref_h_index(v) for k, v in random_cases.items()},
        'i10_index': {k: ref_i10_index(v) for k, v in random_cases.items()},
        'e_index': {k: ref_e_index(v) for k, v in random_cases.items()},
    }
)

diffs = {
    col: int((~np.isclose(impl_r[col], ref_r[col], atol=1e-9)).sum())
    for col in ['h_index', 'i10_index', 'e_index']
}
print(diffs)
print('Máx. desvío e_index:', float((impl_r['e_index'] - ref_r['e_index']).abs().max()))

{'h_index': 0, 'i10_index': 0, 'e_index': 0}
Máx. desvío e_index: 5.684341886080802e-14


## 6. Los dos «e-index»: e-index.net vs. Zhang (2009)

El repo implementa el de **e-index.net**, que es el que declara la referencia del
benchmark (`GTBuilder/APS/code/libs/scholar.py::compute_e_index`):

    e = -1/N · Σ cᵢ·log(cᵢ/c_total)        N = papers citados

En cienciometría, en cambio, «e-index» suele referirse al de **Zhang (2009)**:
las citas excedentes que el h-index tira a la basura,

    e = √(Σ_{j≤h} cⱼ − h²)

Son métricas distintas con el mismo nombre. Esta celda calcula ambas para ver
cuánto se separan — NO para sustituir una por otra.

In [15]:
def zhang_e_index(citations):
    """Zhang (2009): raíz del exceso de citas dentro del h-core."""
    c = sorted(citations, reverse=True)
    h = ref_h_index(c)
    return math.sqrt(sum(c[:h]) - h ** 2)


both = pd.DataFrame(
    {
        'h_index': {k: ref_h_index(v) for k, v in CASES.items()},
        'e_net': {k: ref_e_index(v) for k, v in CASES.items()},
        'e_zhang': {k: zhang_e_index(v) for k, v in CASES.items()},
        'c_total': {k: sum(v) for k, v in CASES.items()},
    }
)
both

,h_index,e_net,e_zhang,c_total
clasico,4,9.097267,3.316625,30
un_paper_famoso,1,-0.000000,9.949874,100
plano,5,8.047190,0.000000,25
sin_citas,0,0.000000,0.000000,0
con_no_citados,3,8.135931,3.741657,23
concentrado,1,4.215017,9.949874,103
repartido,4,35.693415,9.327379,103
cola_larga,6,20.952117,8.717798,119
desordenado,4,9.097267,3.316625,30
calculado_previo,2,3.182571,1.732051,9


In [16]:
# ¿Ordenan igual a los autores? Correlación de rangos sobre los 500 aleatorios.
from scipy.stats import spearmanr

e_net_r = pd.Series({k: ref_e_index(v) for k, v in random_cases.items()})
e_zhang_r = pd.Series({k: zhang_e_index(v) for k, v in random_cases.items()})

rho = spearmanr(e_net_r, e_zhang_r).statistic
print(f'Spearman(e_net, e_zhang) = {rho:.4f}')
print('No son intercambiables: mismo nombre, métricas distintas.')

Spearman(e_net, e_zhang) = 0.9492
No son intercambiables: mismo nombre, métricas distintas.


## 7. Autores reales de OpenAlex

Hasta aquí todo fueron listas inventadas. Ahora los mismos tres índices sobre
autores de verdad, contra dos referencias independientes:

1. **El pipeline** (`eindex.parquet`) — debe coincidir exacto: es la misma función.
2. **OpenAlex oficial** (`summary_stats.parquet`) — NO tiene por qué coincidir
   exacto: OpenAlex calcula sobre el corpus completo del autor y nosotros sobre
   los works que quedaron en el snapshot.

Se usan IDs de OpenAlex, sin nombres.

In [17]:
import duckdb

STATS_DIR = '/data/datasets/LLMScholar-Personas/data/openalex_author_stats'
N_AUTHORS = 5000   # muestra determinista (hash del id), subir si se quiere

con = duckdb.connect()

pairs = con.execute(f"""
    select p.author_url, p.c
    from '{STATS_DIR}/author_work_citations.parquet' p
    join (
        select author_url from '{STATS_DIR}/eindex.parquet'
        order by hash(author_url) limit {N_AUTHORS}
    ) s using(author_url)
""").df()

print(f'{pairs.author_url.nunique():,} autores, {len(pairs):,} pares paper-citas')

5,000 autores, 436,107 pares paper-citas


In [18]:
# Recalcular con la función del pipeline
codes, uniq = pd.factorize(pairs.author_url)
real = _scholarly_stats_from_citations(
    codes.astype(np.int64), pairs.c.values.astype(np.int64), len(uniq)
)
real.index = uniq
real.describe()

,h_index,i10_index,e_index
count,5000.000000,5000.000000,5000.000000
mean,15.251000,33.182600,86.907094
std,19.449279,76.800001,148.370125
min,0.000000,0.000000,0.000000
25%,2.000000,1.000000,4.424085
50%,8.000000,7.000000,42.082391
75%,22.000000,34.250000,113.123892
max,220.000000,1879.000000,3219.717672


### 7.1 Contra el pipeline — debe ser idéntico

In [19]:
pipe = con.execute(f"""
    select author_url, h_index_calc, i10_index_calc, e_index as e_pipeline
    from '{STATS_DIR}/eindex.parquet'
""").df().set_index('author_url')

j = real.join(pipe, how='inner')
print('h dif  :', int((j.h_index != j.h_index_calc).sum()))
print('i10 dif:', int((j.i10_index != j.i10_index_calc).sum()))
print('e max|Δ|:', float((j.e_index - j.e_pipeline).abs().max()))

h dif  : 0
i10 dif: 0
e max|Δ|: 1.7053025658242404e-12


### 7.2 Contra OpenAlex oficial — se espera nuestro <= OA

OpenAlex publica h_index / i10_index calculados sobre TODO el corpus del autor.
Nuestro snapshot es un subconjunto, así que un índice nuestro sólo puede quedar
igual o por debajo. Si apareciera algún autor con nuestro valor POR ENCIMA del
de OpenAlex, eso sí sería un bug de fórmula: no hay forma de sacar más h-index
de menos papers.

In [20]:
oa = con.execute(f"""
    select id as author_url, h_index as h_oa, i10_index as i10_oa
    from '{STATS_DIR}/summary_stats.parquet'
""").df().set_index('author_url')

k = j.join(oa, how='inner')
print(f'autores comparables: {len(k):,}')
for col, ours in [('h', 'h_index'), ('i10', 'i10_index')]:
    same = (k[ours] == k[f'{col}_oa']).mean()
    never_above = (k[ours] <= k[f'{col}_oa']).mean()
    delta = (k[ours] - k[f'{col}_oa'])
    print(f'{col:4} igual: {same:6.1%} | nunca por encima de OA: {never_above:6.1%} '
          f'| Δ min {int(delta.min())}, max {int(delta.max())}')

autores comparables: 5,000
h    igual:  99.0% | nunca por encima de OA: 100.0% | Δ min -13, max 0
i10  igual:  97.3% | nunca por encima de OA: 100.0% | Δ min -149, max 0


In [21]:
# Un valor nuestro por encima del de OpenAlex sería imposible: lo listamos.
impossible = k[(k.h_index > k.h_oa) | (k.i10_index > k.i10_oa)]
if impossible.empty:
    print('Ningún autor supera a OpenAlex. Consistente.')
else:
    print(f'{len(impossible)} autores imposibles:')
    display(impossible[['h_index', 'h_oa', 'i10_index', 'i10_oa']])

Ningún autor supera a OpenAlex. Consistente.


## 8. Autores concretos, uno por uno

`get_citations` saca las citas de cada paper de un autor; `compute` se las pasa
a la función del pipeline. Sirven para inspeccionar casos individuales a mano.

In [22]:
def get_citations(authors, con=con):
    """IDs de OpenAlex -> {author_url: array de citas por paper, desc}.

    Acepta 'A5016309759' o la URL completa, un id suelto o una lista.
    """
    ids = [authors] if isinstance(authors, str) else list(authors)
    ids = [a if a.startswith('http') else f'https://openalex.org/{a}' for a in ids]
    lst = "','".join(ids)
    df = con.execute(f"""
        select author_url, c
        from '{STATS_DIR}/author_work_citations.parquet'
        where author_url in ('{lst}')
        order by author_url, c desc
    """).df()
    return {a: g.c.values.astype(np.int64) for a, g in df.groupby('author_url')}


def compute(citations: dict) -> pd.DataFrame:
    """{author: [citas]} -> h_index / i10_index / e_index, vía el pipeline."""
    idx, cits, names = [], [], list(citations)
    for i, a in enumerate(names):
        idx.extend([i] * len(citations[a]))
        cits.extend(citations[a])
    out = _scholarly_stats_from_citations(
        np.array(idx, dtype=np.int64), np.array(cits, dtype=np.int64), len(names)
    )
    out.index = names
    return out

### 8.1 Cuatro autores de distinto tamaño

In [23]:
AUTHORS = [
    'A5016309759',   # 3 papers, casi sin citas
    'A5065485710',   # 3 papers
    'A5103440466',   # 8 papers, un hit enorme
    'A5103168967',   # 390 papers, carrera larga
]

cit = get_citations(AUTHORS)
for a, v in cit.items():
    print(f"{a.split('/')[-1]}: {len(v)} papers, {(v > 0).sum()} citados, {v.sum():,} citas")
    print(f"   {v[:15].tolist()}{' ...' if len(v) > 15 else ''}\n")

A5016309759: 3 papers, 2 citados, 9 citas
   [8, 1, 0]

A5065485710: 3 papers, 2 citados, 16 citas
   [14, 2, 0]

A5103168967: 390 papers, 314 citados, 10,410 citas
   [381, 302, 239, 235, 212, 194, 188, 175, 149, 147, 146, 145, 142, 142, 139] ...

A5103440466: 8 papers, 5 citados, 692 citas
   [470, 134, 37, 28, 23, 0, 0, 0]



In [27]:
print(cit.get('https://openalex.org/A5103168967').tolist())

[381, 302, 239, 235, 212, 194, 188, 175, 149, 147, 146, 145, 142, 142, 139, 135, 116, 116, 115, 114, 112, 108, 102, 93, 92, 92, 88, 82, 81, 81, 78, 78, 76, 76, 75, 74, 74, 72, 70, 70, 69, 68, 66, 64, 61, 61, 61, 61, 61, 59, 58, 58, 56, 55, 54, 53, 52, 52, 52, 52, 51, 51, 51, 48, 47, 47, 47, 46, 44, 43, 43, 43, 43, 43, 43, 43, 43, 42, 42, 40, 40, 39, 38, 38, 38, 37, 37, 37, 37, 36, 36, 35, 35, 35, 35, 34, 34, 34, 34, 33, 32, 32, 31, 31, 30, 30, 30, 30, 29, 29, 29, 29, 28, 28, 27, 27, 27, 27, 27, 26, 26, 25, 25, 25, 24, 24, 24, 24, 23, 23, 23, 23, 22, 22, 22, 22, 22, 21, 21, 21, 21, 20, 20, 20, 20, 20, 20, 19, 19, 19, 19, 19, 19, 19, 18, 18, 18, 17, 17, 17, 17, 17, 17, 17, 17, 17, 17, 16, 16, 16, 16, 16, 15, 14, 14, 14, 14, 14, 13, 13, 13, 13, 13, 13, 13, 13, 12, 12, 12, 12, 12, 12, 11, 11, 11, 10, 10, 10, 10, 10, 10, 10, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 

In [36]:
matriz = cit.get('https://openalex.org/A5103440466')
no_zeros = matriz[matriz != 0]
total = sum(no_zeros)
long = len(no_zeros)
suma = 0
for val in no_zeros.tolist():
    suma += val*math.log(val/total)
mult = -1/long * suma
        

In [37]:
print(mult)

135.65530511279277


In [25]:
compute(cit)

,h_index,i10_index,e_index
https://openalex.org/A5016309759,1,0,1.569744
https://openalex.org/A5065485710,2,1,3.014161
https://openalex.org/A5103168967,54,202,168.541113
https://openalex.org/A5103440466,5,5,135.655305


### 8.2 Desglose a mano de un autor

Para verificar un caso concreto término por término, sin confiar en la función.

In [26]:
def explain(author, con=con, top=10):
    """Imprime el cálculo paso a paso de los tres índices para UN autor."""
    url = author if author.startswith('http') else f'https://openalex.org/{author}'
    c = get_citations(url)[url]
    c_total, n_cited = int(c.sum()), int((c > 0).sum())

    print(f'{url}')
    print(f'  papers={len(c)}  citados={n_cited}  c_total={c_total}  C_avg={c_total / n_cited:.3f}\n')

    print('  h-index — primer rango donde el paper ya no aguanta el listón:')
    for r, ci in enumerate(c[:top], start=1):
        print(f'    r={r:3}  c={ci:6}  {"OK" if ci >= r else "<- corta aqui"}')
        if ci < r:
            break

    print(f'\n  i10-index — papers con >= 10 citas: {int((c >= 10).sum())}')

    terms = [(ci, ci * math.log(ci / c_total)) for ci in c if ci > 0]
    print(f'\n  e-index — {len(terms)} terminos, primeros {min(top, len(terms))}:')
    for ci, t in terms[:top]:
        print(f'    {ci:6} * ln({ci}/{c_total}) = {t:12.4f}')
    s = sum(t for _, t in terms)
    print(f'    suma = {s:.4f}   e = -1/{n_cited} * suma = {-s / n_cited:.6f}')

    return compute({url: c})


explain('A5103440466')

https://openalex.org/A5103440466
  papers=8  citados=5  c_total=692  C_avg=138.400

  h-index — primer rango donde el paper ya no aguanta el listón:
    r=  1  c=   470  OK
    r=  2  c=   134  OK
    r=  3  c=    37  OK
    r=  4  c=    28  OK
    r=  5  c=    23  OK
    r=  6  c=     0  <- corta aqui

  i10-index — papers con >= 10 citas: 5

  e-index — 5 terminos, primeros 5:
       470 * ln(470/692) =    -181.8210
       134 * ln(134/692) =    -219.9940
        37 * ln(37/692) =    -108.3607
        28 * ln(28/692) =     -89.8067
        23 * ln(23/692) =     -78.2941
    suma = -678.2765   e = -1/5 * suma = 135.655305


,h_index,i10_index,e_index
https://openalex.org/A5103440466,5,5,135.655305
